# 2026-09-07 批次 Vue/San 组件人工验证问题汇总

- 报告日期：2026-09-10
- 批次设计矩阵：`data/datasets/features/design_matrix_2026-09-07_batch_39.json`
- 批次规模：39 对 Vue/San 组件（simple、medium、complex 各 13 对）
- 人工检查覆盖：39/39 对，覆盖率 100%
- 生产缺陷：5 条，影响 5/39 个组件（12.8%）
- 修复后人工复核：5/5 条，复核率 100%

> 证据边界：组件通过状态及缺陷修复结果来自本批串行人工检查中的用户明确确认；最终源码与生成脚本用于静态核对。仓库中未找到可直接匹配本批组件的结构化人工验证日志，因此本报告不能替代自动化浏览器回归测试，也不把静态断言表述为运行时交互测试通过。


## 1. 批次范围与组件清单

| 复杂度 | 数量 | 组件 |
|---|---:|---|
| Simple | 13 | ClipboardSnippet、StarRatingInput、PriceStepper、ReadingProgressCard、OtpCodeInput、ThemePreviewToggle、StockReservationBadge、TipSplitCalculator、ColorContrastBadge、SessionTimeoutNotice、TemperatureDial、FileDropIndicator、BookmarkToggleCard |
| Medium | 13 | ProductFilterPanel、HabitWeekTracker、InvoiceLineEditor、KanbanColumnBoard、MeetingPollScheduler、ExpenseSplitLedger、ImageCropControls、CourseModuleAccordion、DeliveryRouteTimeline、FormRulePlayground、DataPaginationTable、ColorPaletteBuilder、AudioSegmentMarker |
| Complex | 13 | WorkflowDiagramEditor、PolicyRuleComposer、FleetDispatchConsole、ClinicalTriageBoard、DependencyReleasePlanner、EnergyLoadScheduler、AuctionControlRoom、ResearchAnnotationStudio、ProcurementBidMatrix、WarehousePickingWave、SubscriptionRevenueModeler、ApiContractWorkbench、CrisisCommunicationHub |


In [ ]:
from collections import Counter

components_by_level = {
    'simple': ['ClipboardSnippet', 'StarRatingInput', 'PriceStepper', 'ReadingProgressCard', 'OtpCodeInput', 'ThemePreviewToggle', 'StockReservationBadge', 'TipSplitCalculator', 'ColorContrastBadge', 'SessionTimeoutNotice', 'TemperatureDial', 'FileDropIndicator', 'BookmarkToggleCard'],
    'medium': ['ProductFilterPanel', 'HabitWeekTracker', 'InvoiceLineEditor', 'KanbanColumnBoard', 'MeetingPollScheduler', 'ExpenseSplitLedger', 'ImageCropControls', 'CourseModuleAccordion', 'DeliveryRouteTimeline', 'FormRulePlayground', 'DataPaginationTable', 'ColorPaletteBuilder', 'AudioSegmentMarker'],
    'complex': ['WorkflowDiagramEditor', 'PolicyRuleComposer', 'FleetDispatchConsole', 'ClinicalTriageBoard', 'DependencyReleasePlanner', 'EnergyLoadScheduler', 'AuctionControlRoom', 'ResearchAnnotationStudio', 'ProcurementBidMatrix', 'WarehousePickingWave', 'SubscriptionRevenueModeler', 'ApiContractWorkbench', 'CrisisCommunicationHub'],
}

issues = [
    {'id': 'D1', 'category': 'production_defect', 'component': 'StarRatingInput', 'level': 'simple', 'framework': 'san', 'root_cause_group': 'template_iteration', 'verification': 'confirmed_by_user'},
    {'id': 'D2', 'category': 'production_defect', 'component': 'KanbanColumnBoard', 'level': 'medium', 'framework': 'san', 'root_cause_group': 'computed_projection', 'verification': 'confirmed_by_user'},
    {'id': 'D3', 'category': 'production_defect', 'component': 'EnergyLoadScheduler', 'level': 'complex', 'framework': 'san', 'root_cause_group': 'dynamic_class_binding', 'verification': 'confirmed_by_user'},
    {'id': 'D4', 'category': 'production_defect', 'component': 'WarehousePickingWave', 'level': 'complex', 'framework': 'shared', 'root_cause_group': 'boolean_attribute_contract', 'verification': 'confirmed_by_user'},
    {'id': 'D5', 'category': 'production_defect', 'component': 'ApiContractWorkbench', 'level': 'complex', 'framework': 'shared', 'root_cause_group': 'identifier_uniqueness', 'verification': 'confirmed_by_user'},
]

planned_pairs = sum(map(len, components_by_level.values()))
checked_pairs = 39
defects = [item for item in issues if item['category'] == 'production_defect']
affected_components = {item['component'] for item in defects}
confirmed_repairs = sum(item['verification'] == 'confirmed_by_user' for item in defects)
metrics = {
    'planned_pairs': planned_pairs,
    'checked_pairs': checked_pairs,
    'manual_coverage': checked_pairs / planned_pairs,
    'production_defects': len(defects),
    'affected_components': len(affected_components),
    'affected_component_rate': len(affected_components) / planned_pairs,
    'confirmed_repairs': confirmed_repairs,
    'repair_recheck_rate': confirmed_repairs / len(defects),
    'by_level': dict(Counter(item['level'] for item in defects)),
    'by_framework': dict(Counter(item['framework'] for item in defects)),
    'by_root_cause': dict(Counter(item['root_cause_group'] for item in defects)),
}
assert metrics['planned_pairs'] == 39
assert metrics['production_defects'] == 5
assert metrics['affected_components'] == 5
assert metrics['confirmed_repairs'] == 5
metrics


## 2. 问题总览

| ID | 分类 | 组件 | 复杂度 | 框架范围 | 首次人工检查现象 | 当前状态 |
|---|---|---|---|---|---|---|
| D1 | production_defect | StarRatingInput | simple | San | 星星图形未显示 | 已修复，用户确认 |
| D2 | production_defect | KanbanColumnBoard | medium | San | 任务数量渲染为 `[Object]` | 已修复，用户确认 |
| D3 | production_defect | EnergyLoadScheduler | complex | San | 点击运行时段后文字变为“开”，颜色未同步 | 已修复，用户确认 |
| D4 | production_defect | WarehousePickingWave | complex | Vue/San 共有 | 进度 100%、异常 0 时仍禁止完成波次 | 已修复，用户确认 |
| D5 | production_defect | ApiContractWorkbench | complex | Vue/San 共有 | 新增参数后两行输入框同步修改 | 已修复，用户确认 |

本批没有证据表明修复过程引入了独立的 `repair_regression`。测试加载环境问题单列为剩余风险，不计入生产缺陷率。


### D1：StarRatingInput 的 San 数字循环不渲染

- 触发操作：打开 San 测试页观察默认评分控件。
- 现象：评分文字存在，但星星按钮没有显示。
- 根因：模板直接让 `s-for` 遍历数字；该写法在 San 中不能稳定产生指定数量的循环节点。
- 修复：增加 `starValues` 计算属性，将最大星数投影为 `[1, ..., maxStars]`，Vue 与 San 模板都遍历该数组。
- 变更文件：`StarRatingInput.vue`、`StarRatingInput.san`、`scripts/generate_dataset_batch_20260907.js`。
- 验证：最终源码与生成源均包含数组投影；用户在修复后明确确认通过。


### D2：KanbanColumnBoard 的任务数量显示为 `[Object]`

- 触发操作：打开 San 看板并查看各列标题中的任务数。
- 现象：任务数量没有显示为数字，而是 `[Object]`。
- 根因：San 模板直接读取方法返回值并继续访问 `.length`，同时直接遍历方法调用结果，模板表达式语义不稳定。
- 修复：建立 `boardColumns` 计算投影，每列预先生成 `{tasks, count}`；模板只读取 `column.count` 并遍历 `column.tasks`。
- 变更文件：`KanbanColumnBoard.vue`、`KanbanColumnBoard.san`、`scripts/generate_dataset_batch_20260907.js`。
- 验证：最终源码与生成源均采用同一投影结构；用户在修复后明确确认通过。


### D3：EnergyLoadScheduler 状态文字与颜色不同步

- 触发操作：在 San 组件中点击任一运行时段。
- 现象：按钮文字可以切换为“开”，但背景颜色不变化。
- 根因：文字直接依赖 `schedule[device.id][hour]`，类名却通过模板方法隐式读取同一状态；San 没有及时重新求值该模板方法。
- 修复：类名改为直接表达式 `schedule[device.id][hour] ? 'on' : 'off'`，使文字和视觉状态依赖同一响应式路径。
- 变更文件：`EnergyLoadScheduler.san`、`scripts/dataset_batch_20260907_complex.js`。
- 验证：最终 San 模板及复杂组件生成源均包含直接类绑定；用户在修复后明确确认通过。


### D4：WarehousePickingWave 的完成按钮错误禁用

- 触发操作：完成当前波次全部拣选任务，并确保未解决异常数为 0。
- 现象：界面显示进度 100%、异常 0，但“完成波次”仍处于禁用状态。
- 根因：禁用表达式使用 `waveProgress < 100 || unresolvedExceptions.length`。无异常时结果为数字 `0` 而不是严格布尔值 `false`；对 HTML 布尔属性而言，非布尔返回值可能导致属性仍被保留。
- 修复：Vue/San 模板及 `completeWave()` 防御校验统一改为 `unresolvedExceptions.length > 0`。
- 变更文件：`WarehousePickingWave.vue`、`WarehousePickingWave.san`、`scripts/dataset_batch_20260907_complex.js`。
- 验证：三个位置均存在显式布尔比较；用户在修复后明确确认通过。


### D5：ApiContractWorkbench 新增参数发生联动编辑

- 触发操作：在 Vue 组件中新增参数，再编辑新增行或已有行。
- 现象：两行输入框同步修改，无法独立编辑。
- 根因：已有参数 ID 为 10，而 `nextParameterId` 也从 10 开始；新增项产生重复 ID，按 ID 更新时同时命中两行，Vue 的 `:key` 也重复。
- 修复：初始化参数时扫描已分配 ID，以 `max(existing_id) + 1` 计算下一唯一 ID。默认两个 endpoint 的参数 ID 为 10、20，新增参数从 21 开始。
- 变更文件：`ApiContractWorkbench.vue`、`ApiContractWorkbench.san`、`scripts/dataset_batch_20260907_complex.js`。
- 验证：两端最终源码与生成源都包含唯一 ID 推导逻辑；用户在修复后明确确认通过。


## 3. 符合预期但容易误解的行为

### ProcurementBidMatrix：权重 101 时无法选定供应商

用户观察到权重为 101 且无法选定供应商。静态核对表明组件默认权重为 `40 + 25 + 15 + 20 = 100`；界面要求总权重严格等于 100，并要求供应商得分不少于 70。用户调整权重后总和为 101，按钮禁用符合既定业务约束，因此归类为 `expected_behavior`，不计入生产缺陷。

可用性改进建议：在禁用按钮附近明确显示“权重总和必须等于 100”和当前差值，或提供自动归一化/恢复默认权重操作。该建议本批未实施。


## 4. 根因分布与可复用修复模式

五条缺陷分别对应五类根因，各 1 条：模板数字循环、模板方法/派生对象、动态类的间接依赖、布尔属性类型契约、列表标识符唯一性。样本量较小，不能据此断言一般性分布，但可以提炼以下迁移规则：

1. **循环数据先标准化**：不要假设 Vue 与 San 对数字循环或方法返回集合有相同语义；先构造明确数组或计算投影。
2. **同一状态使用同一直接依赖**：文本、类名和禁用状态应直接依赖同一响应式字段，避免模板方法隐藏依赖。
3. **布尔属性返回严格布尔值**：`disabled`、`checked`、`selected` 等绑定统一使用显式比较或 `Boolean(...)`。
4. **动态列表 ID 从现存数据推导**：计数器不能与种子数据硬编码 ID 冲突；初始化后取最大值加一，并保证渲染 key 与更新定位使用同一唯一键。
5. **补丁同步回生成源**：只修成品会在下一次批量生成时被覆盖；组件源码与相应生成脚本必须同时修正。


In [ ]:
import json
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'data/datasets/features/design_matrix_2026-09-07_batch_39.json').is_file():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

repo_root = find_repo_root()
matrix_path = repo_root / 'data/datasets/features/design_matrix_2026-09-07_batch_39.json'
matrix = json.loads(matrix_path.read_text(encoding='utf-8'))
matrix_pairs = [(item['component_name'], item['level']) for item in matrix['components']]
expected_pairs = [(name, level) for level, names in components_by_level.items() for name in names]
assert matrix['batch_size'] == 39
assert matrix_pairs == expected_pairs

level_dirs = {'simple': '01_simple', 'medium': '02_medium', 'complex': '03_complex'}
missing = []
for name, level in matrix_pairs:
    base = repo_root / 'data/datasets/components' / level_dirs[level] / name
    for framework, suffix in [('vue', '.vue'), ('san', '.san')]:
        path = base / framework / f'{name}{suffix}'
        if not path.is_file():
            missing.append(str(path.relative_to(repo_root)))
assert not missing, missing
{'matrix_pairs': len(matrix_pairs), 'component_files': len(matrix_pairs) * 2, 'missing': missing}


In [ ]:
def read(relative_path):
    return (repo_root / relative_path).read_text(encoding='utf-8')

source_checks = {
    'D1_san_array_iteration': ('data/datasets/components/01_simple/StarRatingInput/san/StarRatingInput.san', ['s-for=\"star in starValues', 'starValues()']),
    'D1_generator_synced': ('scripts/generate_dataset_batch_20260907.js', ['starValues', 'Array.from({ length: this.maxStars }']),
    'D2_san_projection': ('data/datasets/components/02_medium/KanbanColumnBoard/san/KanbanColumnBoard.san', ['boardColumns', 'column.count', 'column.tasks']),
    'D2_generator_synced': ('scripts/generate_dataset_batch_20260907.js', ['boardColumns', 'count: tasks.length']),
    'D3_direct_class_binding': ('data/datasets/components/03_complex/EnergyLoadScheduler/san/EnergyLoadScheduler.san', ["schedule[device.id][hour] ? 'on' : 'off'"]),
    'D3_generator_synced': ('scripts/dataset_batch_20260907_complex.js', [":class=\"schedule[device.id][hour] ? 'on' : 'off'\""]),
    'D4_vue_strict_boolean': ('data/datasets/components/03_complex/WarehousePickingWave/vue/WarehousePickingWave.vue', ['unresolvedExceptions.length > 0']),
    'D4_san_strict_boolean': ('data/datasets/components/03_complex/WarehousePickingWave/san/WarehousePickingWave.san', ['unresolvedExceptions.length > 0']),
    'D4_generator_synced': ('scripts/dataset_batch_20260907_complex.js', ['unresolvedExceptions.length > 0']),
    'D5_vue_unique_id': ('data/datasets/components/03_complex/ApiContractWorkbench/vue/ApiContractWorkbench.vue', ['let nextParameterId = 1', 'Math.max(nextParameterId, parameterId + 1)']),
    'D5_san_unique_id': ('data/datasets/components/03_complex/ApiContractWorkbench/san/ApiContractWorkbench.san', ['let nextParameterId = 1', 'Math.max(nextParameterId, parameterId + 1)']),
    'D5_generator_synced': ('scripts/dataset_batch_20260907_complex.js', ['let nextParameterId = 1', 'Math.max(nextParameterId, parameterId + 1)']),
}
check_results = {}
for check_name, (relative_path, fragments) in source_checks.items():
    content = read(relative_path)
    missing_fragments = [fragment for fragment in fragments if fragment not in content]
    check_results[check_name] = not missing_fragments
    assert not missing_fragments, (check_name, relative_path, missing_fragments)
check_results


## 5. 验证证据与可报告指标

批次结构校验命令：

```powershell
node scripts/validate_dataset_batch_20260902.js design_matrix_2026-09-07_batch_39.json 39 13
```

已记录的校验结果为：batch 39；simple/medium/complex 各 13；manifest 总量 100；加载脚本 78；errors 与 warnings 均为空。此结果证明批次数量、文件与生成结构满足校验器约束，不证明所有运行时交互正确。

| 指标 | 结果 | 统计口径 |
|---|---:|---|
| 人工检查覆盖率 | 39/39 = 100% | Vue 与 San 均完成串行人工检查并获得确认的组件对 / 计划组件对 |
| 生产缺陷数 | 5 | 仅统计 `production_defect` |
| 受影响组件率 | 5/39 = 12.8% | 有至少一条生产缺陷的去重组件 / 批次组件对 |
| 修复后人工复核率 | 5/5 = 100% | 用户明确确认通过的已修复生产缺陷 / 已修复生产缺陷 |
| 复杂度分布 | simple 1、medium 1、complex 3 | 按受影响组件去重统计 |
| 框架范围分布 | San 3、shared 2 | shared 不重复计入 Vue 与 San |
| 符合预期行为 | 1 | 单独报告，不计入生产缺陷率 |
| 修复回归 | 0 条已识别 | 当前证据未发现，不等同于自动化证明不存在 |


## 6. 剩余风险与后续行动

| 风险 | 当前证据 | 影响 | 建议 |
|---|---|---|---|
| Runner 依赖外部 CDN | Vue 测试页曾出现 `https://unpkg.com/http-vue-loader` 超时，随后 `httpVueLoader is not defined` | 网络不稳定时组件无法加载，会干扰缺陷判断 | 将 Vue、San 与加载器固定到本地依赖，增加明确的加载失败提示 |
| 浏览器扩展注入噪声 | 控制台出现 `content_main.js ... classList`，调用栈来自注入脚本 | 容易被误归因到组件源码 | 使用无扩展浏览器配置复核，并按调用栈来源区分环境错误 |
| 缺少自动化交互回归 | 当前通过结论主要来自逐组件人工确认 | 后续重新生成或依赖升级可能让缺陷复发 | 为五种已知失败模式补充最小浏览器交互测试 |
| Runner 与正式构建不同 | 手工页面直接加载组件/框架，不能覆盖正式打包链路 | 可能遗漏编译、依赖解析与生产模式差异 | 增加正式构建产物的冒烟测试 |
| 缺少结构化人工验证日志 | `migration_notes.json` 未找到本批组件的可匹配记录 | 复核时间与步骤依赖会话上下文，追溯性有限 | 后续每次确认同步写入批次验证 JSON，再由报告技能读取 |


## 7. 论文可用结论

在本批 39 对 Vue/San 成对组件的串行人工验证中，计划样本全部完成检查。共识别 5 条生产后缺陷，涉及 5 个组件，批内受影响组件率为 12.8%（5/39）；其中 San 特有缺陷 3 条，共享实现缺陷 2 条。全部 5 条缺陷在修复后得到用户人工确认。缺陷主要暴露在框架间模板表达式语义、响应式类绑定、HTML 布尔属性类型以及动态列表标识符管理方面。结果说明，语法级迁移成功和批次结构校验通过不足以替代交互级验证，迁移数据生产流程需要同时约束模板表达式、状态依赖和数据身份。

该结论仅描述本次目的性生成样本，不能外推为一般 Vue-to-San 迁移缺陷率。人工判断可能受到测试人员、浏览器环境和 runner 实现影响；缺少自动化交互回归与结构化时间戳也限制了可重复性。后续实验应保留同一检查协议，记录逐组件机器可读结果，并对已出现的五类失败模式建立自动化测试。


## 8. 文件索引

- 设计矩阵：`data/datasets/features/design_matrix_2026-09-07_batch_39.json`
- 数据集 manifest：`data/datasets/dataset_manifest.json`
- 组件目录：`data/datasets/components/01_simple`、`02_medium`、`03_complex`
- Simple/Medium 生成源：`scripts/generate_dataset_batch_20260907.js`
- Complex 生成源：`scripts/dataset_batch_20260907_complex.js`
- 批次校验器：`scripts/validate_dataset_batch_20260902.js`
- 人工测试页：`tests/manual/vue-test-runner.html`、`tests/manual/san-test-runner.html`
- 报告技能：`skills/vue-san-validation-report/SKILL.md`
- 证据口径：`skills/vue-san-validation-report/references/validation-evidence-schema.md`
